# TFT mit **gelaggten** Exogenen (realistisch, kein Zukunftswetter)

Gegenstück zur Perfect-Prognosis-Auswertung: Wetter (und optional Co-Schadstoffe) gehen **nur als `hist_exog`** ein — das Modell sieht sie **nur in der Vergangenheit** (bis zum Vorhersage-Ursprung), aber **nicht** über den Horizont. Damit fließt **kein Zukunftswetter** ein (analog zu Prophet 12.2/12.3, Lag/Persistenz).

- **`futr_exog` (im Voraus bekannt):** Zeit-Features (Stunde/Tag) + Feiertag.
- **`hist_exog` (nur Vergangenheit):** Wetter, optional + Co-Schadstoffe.
- Split/Metriken/Rolling **identisch** zu Prophet (MAE/RMSE/MASE/MAPE bei 8/24/48/72 h).

Kernel **Python (tft)**, Run All. Schreibt `ergebnis_tft_lag.csv` und `ergebnis_tft_lag_allstations.csv`.

In [1]:
import warnings; warnings.filterwarnings('ignore')
import os, gc
import numpy as np
import pandas as pd
import torch
import holidays
from pathlib import Path
from neuralforecast import NeuralForecast
from neuralforecast.models import TFT
from neuralforecast.losses.pytorch import MAE as NF_MAE

torch.set_float32_matmul_precision('high')

STATION       = 'Aotizhongxin'
FREQ          = 'h'
H_TFT         = 72
INPUT_SIZE    = 168
MAX_STEPS     = 1000
BATCH_SIZE    = 32
HIDDEN_SIZE   = 64
WINDOWS_BATCH = 128        # bei CUDA-OOM weiter senken (64/32)
HORIZONTE     = [8, 24, 48, 72]
DATA_DIR      = Path('../data/prepared')
RAW_DIR       = Path('../data/PRSA_Data_20130301-20170228')
REGRESSOREN   = ['TEMP', 'DEWP', 'PRES', 'WSPM', 'RAIN', 'wd_sin', 'wd_cos']
CO_POLL       = ['PM10', 'SO2', 'NO2', 'CO', 'O3']

print('GPU:', torch.cuda.is_available())

GPU: True


In [2]:
def lade(variante, station=STATION):
    tr = pd.read_csv(DATA_DIR / variante / f'prophet_train_{station}.csv', parse_dates=['ds'])
    te = pd.read_csv(DATA_DIR / variante / f'prophet_test_{station}.csv',  parse_dates=['ds'])
    return tr, te

def lade_coschadstoffe(station):
    d = pd.read_csv(RAW_DIR / f'PRSA_Data_{station}_20130301-20170228.csv')
    d['ds'] = pd.to_datetime(d[['year','month','day','hour']])
    s = d[['ds'] + CO_POLL].sort_values('ds').set_index('ds')
    s = s.reindex(pd.date_range(s.index.min(), s.index.max(), freq='h'))
    s[CO_POLL] = s[CO_POLL].interpolate(method='time', limit_direction='both')
    return s.reset_index().rename(columns={'index':'ds'})

def add_feiertag(df):
    cn = holidays.country_holidays('CN', years=range(2013, 2018))
    d = df.copy(); d['feiertag'] = d['ds'].dt.date.astype('O').map(lambda t: 1 if t in cn else 0); return d

def add_time_features(df):
    d = df.copy()
    d['hour_sin'] = np.sin(2*np.pi*d['ds'].dt.hour/24); d['hour_cos'] = np.cos(2*np.pi*d['ds'].dt.hour/24)
    d['day_sin']  = np.sin(2*np.pi*d['ds'].dt.dayofyear/365.25); d['day_cos'] = np.cos(2*np.pi*d['ds'].dt.dayofyear/365.25)
    return d

def regularize(df, spalten):
    df = df.copy(); df['ds'] = pd.to_datetime(df['ds'])
    d = df.set_index('ds').sort_index()
    d = d.reindex(pd.date_range(d.index.min(), d.index.max(), freq=FREQ))
    for c in spalten:
        d[c] = d[c].interpolate(limit_direction='both')
    d.index.name = 'ds'; return d.reset_index()

def baue_df(variante, station, hist_regs, feiertag):
    tr, te = lade(variante, station)
    if any(c in CO_POLL for c in hist_regs):
        co = lade_coschadstoffe(station)
        tr = tr.merge(co, on='ds', how='left'); te = te.merge(co, on='ds', how='left')
    skala = mase_skala(tr['y'])
    g = regularize(pd.concat([tr, te], ignore_index=True), spalten=['y'] + hist_regs)
    d = add_time_features(g)
    if feiertag: d = add_feiertag(d)
    d.insert(0, 'unique_id', station)
    futr = ['hour_sin','hour_cos','day_sin','day_cos'] + (['feiertag'] if feiertag else [])
    cols = ['unique_id','ds','y'] + hist_regs + futr
    return d[cols], skala, futr

def mae(y, yh):  return float(np.mean(np.abs(np.asarray(y,float)-np.asarray(yh,float))))
def rmse(y, yh): return float(np.sqrt(np.mean((np.asarray(y,float)-np.asarray(yh,float))**2)))
def mape(y, yh):
    y=np.asarray(y,float); yh=np.asarray(yh,float); m=y>0
    return float(np.mean(np.abs((y[m]-yh[m])/y[m]))*100) if m.any() else np.nan
def mase_skala(train_y, mp=24):
    a=np.asarray(train_y,float); return float(np.nanmean(np.abs(a[mp:]-a[:-mp])))

In [3]:
# --- Einzelstation: TFT mit gelaggten (historischen) Exogenen ---
def tft_lag(variante, hist_regs, feiertag, name, station=STATION):
    df, skala, futr_exog = baue_df(variante, station, hist_regs, feiertag)
    hist_exog = hist_regs                                        # Wetter(+Co) nur historisch
    _, te = lade(variante, station); n_windows = len(te) // H_TFT
    model = TFT(h=H_TFT, input_size=INPUT_SIZE,
                hist_exog_list=hist_exog, futr_exog_list=futr_exog,
                loss=NF_MAE(), max_steps=MAX_STEPS, batch_size=BATCH_SIZE,
                hidden_size=HIDDEN_SIZE, windows_batch_size=WINDOWS_BATCH,
                inference_windows_batch_size=WINDOWS_BATCH,
                accelerator='gpu', devices=1, precision='16-mixed', enable_progress_bar=True)
    nf = NeuralForecast(models=[model], freq=FREQ)
    cv = nf.cross_validation(df=df, n_windows=n_windows, step_size=H_TFT)
    cv['yhat'] = cv['TFT'].clip(lower=0)
    cv['lead'] = ((cv['ds'] - cv['cutoff']).dt.total_seconds() // 3600).astype(int)
    rows = []
    for b in HORIZONTE:
        w = cv[cv['lead'] <= b]
        rows.append({'Modell': name, 'Horizont': f'{b} h',
                     'MAE': mae(w['y'], w['yhat']), 'RMSE': rmse(w['y'], w['yhat']),
                     'MASE': mae(w['y'], w['yhat'])/skala, 'MAPE %': mape(w['y'], w['yhat'])})
    del model, nf, cv; gc.collect(); torch.cuda.empty_cache()
    return pd.DataFrame(rows)

konfigs = {
    'TFT Lag: Wetter':    dict(variante='behandelt', hist_regs=REGRESSOREN,            feiertag=True),
    'TFT Lag: Wetter+Co': dict(variante='behandelt', hist_regs=REGRESSOREN + CO_POLL,  feiertag=True),
}
alle = []
for name, cfg in konfigs.items():
    print('== Trainiere', name, '=='); torch.cuda.empty_cache()
    tab = tft_lag(name=name, **cfg)
    print(tab.round(3).to_string(index=False)); print(); alle.append(tab)

ergebnis_tft_lag = pd.concat(alle, ignore_index=True)[['Modell','Horizont','MAE','RMSE','MASE','MAPE %']].round(3)
print('=== Einzelstation (Lag) ==='); print(ergebnis_tft_lag.to_string(index=False))
ergebnis_tft_lag.to_csv('../data/ergebnis_tft_lag.csv', index=False)
print('gespeichert -> ../data/ergebnis_tft_lag.csv')

== Trainiere TFT Lag: Wetter ==


Seed set to 1
Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name                    | Type                     | Params
---------------------------------------------------------------------
0 | loss                    | MAE                      | 0     
1 | padder_train            | ConstantPad1d            | 0     
2 | scaler                  | TemporalNorm             | 0     
3 | embedding               | TFTEmbedding             | 1.7 K 
4 | temporal_encoder        | TemporalCovariateEncoder | 482 K 
5 | temporal_fusion_decoder | TemporalFusionDecoder    | 64.8 K
6 | output_adapter          | Linear                   | 65    
---------------------------------------------------------------------
548 K     Trainable params
0         Non-trainable params
548 K     Total params
2.194     Tota

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_steps=1000` reached.
Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Predicting: |          | 0/? [00:00<?, ?it/s]

            Modell Horizont    MAE   RMSE  MASE  MAPE %
TFT Lag: Wetter+Co      8 h 29.322 49.399 0.489  86.940
TFT Lag: Wetter+Co     24 h 39.579 64.721 0.660 132.210
TFT Lag: Wetter+Co     48 h 52.520 84.590 0.876 156.314
TFT Lag: Wetter+Co     72 h 56.280 86.963 0.939 171.998

=== Einzelstation (Lag) ===
            Modell Horizont    MAE   RMSE  MASE  MAPE %
   TFT Lag: Wetter      8 h 29.381 51.131 0.490  76.301
   TFT Lag: Wetter     24 h 41.018 66.723 0.684 133.290
   TFT Lag: Wetter     48 h 54.071 85.400 0.902 169.547
   TFT Lag: Wetter     72 h 59.186 89.576 0.988 201.507
TFT Lag: Wetter+Co      8 h 29.322 49.399 0.489  86.940
TFT Lag: Wetter+Co     24 h 39.579 64.721 0.660 132.210
TFT Lag: Wetter+Co     48 h 52.520 84.590 0.876 156.314
TFT Lag: Wetter+Co     72 h 56.280 86.963 0.939 171.998
gespeichert -> ../data/ergebnis_tft_lag.csv


## Alle 12 Stationen (globales TFT, gelaggte Exogene)

Ein globales TFT über alle 12 Stationen, Wetter(+Co) als `hist_exog`. MASE je Station, über 12 Stationen gemittelt. Schreibt `../data/ergebnis_tft_lag_allstations.csv`.

In [4]:
stationen = sorted(p.name.replace('prophet_train_', '').replace('.csv', '')
                   for p in (DATA_DIR / 'basis').glob('prophet_train_*.csv'))

def tft_lag_all(variante, hist_regs, feiertag, name):
    teile, skalen = [], {}
    for st in stationen:
        d, sk, futr_exog = baue_df(variante, st, hist_regs, feiertag)
        skalen[st] = sk; teile.append(d)
    multi = pd.concat(teile, ignore_index=True)
    hist_exog = hist_regs
    _, te0 = lade(variante)
    test_h = int((te0['ds'].max() - te0['ds'].min()) / pd.Timedelta(hours=1)) + 1
    n_windows = test_h // H_TFT
    model = TFT(h=H_TFT, input_size=INPUT_SIZE,
                hist_exog_list=hist_exog, futr_exog_list=futr_exog,
                loss=NF_MAE(), max_steps=MAX_STEPS, batch_size=BATCH_SIZE,
                hidden_size=HIDDEN_SIZE, windows_batch_size=WINDOWS_BATCH,
                inference_windows_batch_size=WINDOWS_BATCH,
                accelerator='gpu', devices=1, precision='16-mixed', enable_progress_bar=True)
    nf = NeuralForecast(models=[model], freq=FREQ)
    cv = nf.cross_validation(df=multi, n_windows=n_windows, step_size=H_TFT)
    cv['yhat'] = cv['TFT'].clip(lower=0)
    cv['lead'] = ((cv['ds'] - cv['cutoff']).dt.total_seconds() // 3600).astype(int)
    rows = []
    for b in HORIZONTE:
        w = cv[cv['lead'] <= b]
        per = [{'MAE': mae(g['y'], g['yhat']), 'RMSE': rmse(g['y'], g['yhat']),
                'MASE': mae(g['y'], g['yhat']) / skalen[st], 'MAPE %': mape(g['y'], g['yhat'])}
               for st, g in w.groupby('unique_id')]
        m = pd.DataFrame(per).mean()
        rows.append({'Modell': name, 'Horizont': f'{b} h', 'MAE': m['MAE'], 'RMSE': m['RMSE'],
                     'MASE': m['MASE'], 'MAPE %': m['MAPE %']})
    del model, nf, cv; gc.collect(); torch.cuda.empty_cache()
    return pd.DataFrame(rows)

konfigs_all = {
    'TFT 12 Stat. Lag: Wetter':    dict(variante='behandelt', hist_regs=REGRESSOREN,           feiertag=True),
    'TFT 12 Stat. Lag: Wetter+Co': dict(variante='behandelt', hist_regs=REGRESSOREN + CO_POLL, feiertag=True),
}
alle_all = []
for name, cfg in konfigs_all.items():
    print('== Globales TFT (Lag) ueber 12 Stationen:', name, '=='); torch.cuda.empty_cache()
    tab = tft_lag_all(name=name, **cfg)
    print(tab.round(3).to_string(index=False)); print(); alle_all.append(tab)

ergebnis_tft_lag_all = pd.concat(alle_all, ignore_index=True)[['Modell','Horizont','MAE','RMSE','MASE','MAPE %']].round(3)
print('=== 12 Stationen (Lag, Mittel) ==='); print(ergebnis_tft_lag_all.to_string(index=False))
ergebnis_tft_lag_all.to_csv('../data/ergebnis_tft_lag_allstations.csv', index=False)
print('gespeichert -> ../data/ergebnis_tft_lag_allstations.csv')

== Globales TFT (Lag) ueber 12 Stationen: TFT 12 Stat. Lag: Wetter ==


Seed set to 1
Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name                    | Type                     | Params
---------------------------------------------------------------------
0 | loss                    | MAE                      | 0     
1 | padder_train            | ConstantPad1d            | 0     
2 | scaler                  | TemporalNorm             | 0     
3 | embedding               | TFTEmbedding             | 1.7 K 
4 | temporal_encoder        | TemporalCovariateEncoder | 482 K 
5 | temporal_fusion_decoder | TemporalFusionDecoder    | 64.8 K
6 | output_adapter          | Linear                   | 65    
---------------------------------------------------------------------
548 K     Trainable params
0         Non-trainable params
548 K     Total params
2.194     Tota

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_steps=1000` reached.
Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Predicting: |          | 0/? [00:00<?, ?it/s]

                     Modell Horizont    MAE   RMSE  MASE  MAPE %
TFT 12 Stat. Lag: Wetter+Co      8 h 29.657 50.088 0.503  91.845
TFT 12 Stat. Lag: Wetter+Co     24 h 41.272 67.472 0.702 128.108
TFT 12 Stat. Lag: Wetter+Co     48 h 53.710 88.049 0.914 147.991
TFT 12 Stat. Lag: Wetter+Co     72 h 59.104 92.503 1.006 178.004

=== 12 Stationen (Lag, Mittel) ===
                     Modell Horizont    MAE   RMSE  MASE  MAPE %
   TFT 12 Stat. Lag: Wetter      8 h 29.361 48.108 0.498  78.930
   TFT 12 Stat. Lag: Wetter     24 h 39.758 64.607 0.677 144.271
   TFT 12 Stat. Lag: Wetter     48 h 51.217 82.163 0.872 186.720
   TFT 12 Stat. Lag: Wetter     72 h 55.455 85.442 0.945 214.549
TFT 12 Stat. Lag: Wetter+Co      8 h 29.657 50.088 0.503  91.845
TFT 12 Stat. Lag: Wetter+Co     24 h 41.272 67.472 0.702 128.108
TFT 12 Stat. Lag: Wetter+Co     48 h 53.710 88.049 0.914 147.991
TFT 12 Stat. Lag: Wetter+Co     72 h 59.104 92.503 1.006 178.004
gespeichert -> ../data/ergebnis_tft_lag_allstations.cs

### Hinweise

- **Kein Leakage:** Wetter/Co stehen zwar als Spalten im df (auch fuer die Testzeit), aber weil sie als `hist_exog` deklariert sind, nutzt TFT sie strukturell **nur im Eingangsfenster** (Vergangenheit) — nie ueber den Horizont. Das ist der saubere Weg fuer gelaggte Exogene.
- **Fairer Vergleich:** Diese Zahlen gehoeren neben Prophet **+ Wetter (Lag)** bzw. **+ Wetter+Co (Lag)** aus 12.2/12.3.
- Bei CUDA-OOM `WINDOWS_BATCH`/`HIDDEN_SIZE` senken; nach OOM Kernel neu starten.